# 05 — Health, social connection and subjective well-being

**Purpose:** evaluate candidate questions with domain-appropriate timing. This is descriptive comparative analysis, not causal inference. Social support and negative affect are six independent three-year pooled windows, not annual observations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.oecd_audit import INDICATOR_SPECS, load_clean

PEERS = ['CAN', 'NZL', 'GBR', 'USA']
PROTOCOLS = {'health_longevity': ('5_1', 2010, 2023), 'health_harm': ('5_3', 2010, 2024), 'social_support': ('7_1_DEP', 2010, 2024), 'negative_affect': ('11_2', 2010, 2024)}
TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'

def endpoint_scorecard(data, code, start, end, references):
    x = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *references]) & data.year.isin([start, end])].sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    w = x.pivot(index='country_code', columns='year', values='value').reindex(columns=[start, end]).dropna()
    change = w[end] - w[start]; oriented = change * (1 if INDICATOR_SPECS[code].direction == 'higher' else -1)
    peer = oriented.drop('AUS'); rank = oriented.rank(ascending=False, method='average')['AUS']; n = len(oriented)
    return {'indicator_code': code, 'indicator': data.loc[data.indicator_code.eq(code), 'indicator'].iloc[0], 'start_year': start, 'end_year': end, 'australia_start_value': w.loc['AUS', start], 'australia_end_value': w.loc['AUS', end], 'australia_absolute_change': change['AUS'], 'australia_oriented_change': oriented['AUS'], 'reference_country_count': len(peer), 'reference_median_oriented_change': peer.median(), 'australia_change_percentile': 100*(n-rank)/(n-1)}

def relative_trend(data, code, start, end):
    x = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *PEERS]) & data.year.between(start, end)].sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    aus = x.loc[x.country_code.eq('AUS'), ['year', 'value']].rename(columns={'value': 'australia_value'})
    peer = x.loc[x.country_code.ne('AUS')].groupby('year', as_index=False).agg(reference_median=('value', 'median'), reference_country_count=('country_code', 'nunique'))
    return aus.merge(peer, on='year')

df = load_clean()
TABLE_DIR.mkdir(parents=True, exist_ok=True)

## Comparative endpoint results

Each outcome uses the latest actual Australian observation through 2024. Life expectancy ends in 2023; pooled social and affect data end in the 2023–25 window displayed as 2024. A higher oriented change is favourable.

In [ ]:
all_references = sorted(set(df.country_code) - {'AUS'})
scorecards = pd.DataFrame([{'analysis': name, **endpoint_scorecard(df, code, start, end, all_references)} for name, (code, start, end) in PROTOCOLS.items()])
scorecards.to_csv(TABLE_DIR / 'health_social_wellbeing_scorecards.csv', index=False)
display(scorecards)

social_trend = relative_trend(df, '7_1_DEP', 2010, 2024)
affect_trend = relative_trend(df, '11_2', 2010, 2024)
social_trend.to_csv(TABLE_DIR / 'social_support_relative_trend_english_peers.csv', index=False)
affect_trend.to_csv(TABLE_DIR / 'negative_affect_relative_trend_english_peers.csv', index=False)
display(social_trend)
display(affect_trend)

## Interpretation

The strongest primary-question extension is whether material progress coexisted with worsening perceived social support and negative affect. The health contrast is compelling but secondary because the adverse mortality measure combines distinct causes and has thin latest-year coverage. See `docs/analysis/health_social_wellbeing_question_assessment.md` for claim boundaries and candidate-question ranking.